In [3]:
import pandas as pd
import numpy as np
import os
import glob
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import gdown
import zipfile
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="keras")

<h2 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Dataset Introduction
</font>
</h2>

<p dir=ltr style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
The provided dataset in JSON format contains information about 1599 accommodations, which include various features. This dataset contains a list with the key "rooms" where the available accommodations are located. Each record in this dataset represents an accommodation with its corresponding characteristics and amenities. In the table below, the keys and their descriptions are provided.
</font>
</p>

<center>
<div dir=ltr style="direction: ltr;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>

|Key|Description|
|:------:|:---:|
|id|Unique identifier of the accommodation|
|title|Title of the accommodation|
|description|Descriptions about the accommodation|
|province|Province information including the province ID and name|
|city|City information including the city ID and name|

</font>
</div>
</center>


<p dir=ltr style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
A sample of this data is as follows:
</font>
</p>

```json
{
  "rooms": [
    {
      "id": 3202100,
      "title": "Suite rental on Javaherdeh Road - Eshkankooh",
      "description": "This studio suite with a spacious balcony and a beautiful view is located on the ground floor of a two-story building. The distance from the accommodation to the supermarket and bakery is approximately 50 and 500 meters, respectively.",
      "province": {
        "id": "p26",
        "name": "Mazandaran"
      },
      "city": {
        "id": 303,
        "name": "Ramsar"
      }
    },
    ...
  ]
}

<h2 align="left" style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Reading the Dataset
</font>
</h2>

<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
    First, you need to read the dataset files. The training features are stored in the <code>train.json</code> file, and the directory related to the images of the test samples for which you must predict their category is stored in the <code>test.json</code> file. If needed, you can separate a portion of the training data as validation data. Please note that the <code>description</code> and <code>title</code> columns have been removed from the test data, and the main purpose of this file is to access the directory of the test data images. The directory of each image is equal to the <code>id</code> of each accommodation and is located inside the <code>pictures</code> folder.
</font>
</p>

<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
    <font face="vazir">
        <span style="color:orange"><b>Note:</b></span>
        The size of the files for this question is approximately 3.8 GB. If loading these files in the Colab environment is difficult for you, you can download these two files directly into the Colab environment using the following command.
    </font>
</p>

In [ ]:
data_url = "https://drive.google.com/file/d/1b8O_a6ywcsbLqJAGDGCkePrdn1cFlXl0/view?usp=sharing"
downloaded_file = gdown.download(data_url, output = None, quiet = False)
extract_folder = '.'
with zipfile.ZipFile(downloaded_file, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)
os.remove(downloaded_file)

Downloading...
From (original): https://drive.google.com/uc?id=1b8O_a6ywcsbLqJAGDGCkePrdn1cFlXl0
From (redirected): https://drive.google.com/uc?id=1b8O_a6ywcsbLqJAGDGCkePrdn1cFlXl0&confirm=t&uuid=b51734d1-d39d-4300-b690-deb7de0e175a
To: /home/norbert/Projects/computer_vision_oly/classification/pool/data_pool.zip
100%|██████████| 4.06G/4.06G [15:36<00:00, 4.34MB/s]


In [5]:
train_data = pd.read_json('train.json')
print(train_data)

                                                  rooms
0     {'id': 3175858, 'title': 'رزرو ویلا با استخر ر...
1     {'id': 3237321, 'title': 'اجاره ویلا استخردار ...
2     {'id': 3154228, 'title': 'اجاره ویلا با استخر ...
3     {'id': 3169850, 'title': 'رزرو سوئیت در جواهرد...
4     {'id': 3207557, 'title': 'اجاره سوئیت در جاده ...
...                                                 ...
1594  {'id': 3167459, 'title': 'اجاره ویلا استخردار ...
1595  {'id': 3207406, 'title': 'اجاره ویلا استخردار ...
1596  {'id': 3172620, 'title': 'اجاره ویلا چابکسر - ...
1597  {'id': 3237948, 'title': 'اجاره سوئیت در جواهر...
1598  {'id': 3229812, 'title': 'اجاره خانه با ویو کو...

[1599 rows x 1 columns]


In [6]:
test_data = pd.read_json('test.json')
print(test_data)

               rooms
0    {'id': 3160664}
1    {'id': 3195184}
2    {'id': 3224078}
3    {'id': 3233712}
4    {'id': 3201449}
..               ...
856  {'id': 3232618}
857  {'id': 3230930}
858  {'id': 3171083}
859  {'id': 3168538}
860  {'id': 3168123}

[861 rows x 1 columns]


In [ ]:
df_train = pd.json_normalize(train_data['rooms'])
df_test = pd.json_normalize(test_data['rooms'])

def extract_pool_label(row):
    text = str(row.get('title', '')) + " " + str(row.get('description', ''))
    
    pool_keywords = ['استخر', 'جکوزی', 'آبگرم', 'آب گرم']
    
    for kw in pool_keywords:
        if kw in text.lower():
            return True
    return False

df_train['Pool'] = df_train.apply(extract_pool_label, axis=1)

print(df_train['Pool'].value_counts())

Pool
True     933
False    666
Name: count, dtype: int64


In [ ]:
TRAIN_DIR = 'train'
TEST_DIR = 'test'

def create_image_dataframe(df, base_dir, has_label=True):
    img_paths = []
    ids = []
    labels = []
    
    for index, row in df.iterrows():
        room_id = str(row['id'])
        room_dir = os.path.join(base_dir, room_id)
        
        if os.path.exists(room_dir):
            for img_name in os.listdir(room_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img_paths.append(os.path.join(room_dir, img_name))
                    ids.append(row['id'])
                    if has_label:
                        labels.append(row['Pool'])
                        
    if has_label:
        return pd.DataFrame({'id': ids, 'image_path': img_paths, 'Pool': labels})
    else:
        return pd.DataFrame({'id': ids, 'image_path': img_paths})

train_img_df = create_image_dataframe(df_train, base_dir=TRAIN_DIR, has_label=True)
test_img_df = create_image_dataframe(df_test, base_dir=TEST_DIR, has_label=False)

print(f"train_img_df : {len(train_img_df)}")
print(f"test_img_df : {len(test_img_df)}")

if len(train_img_df) > 0:
    train_img_df['Pool_str'] = train_img_df['Pool'].astype(str)

train_img_df : 15841
test_img_df : 8505


In [10]:
datagen = ImageDataGenerator(validation_split=0.2, rescale=1./255)

train_generator = datagen.flow_from_dataframe(
    dataframe=train_img_df,
    x_col='image_path',
    y_col='Pool_str',
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_generator = datagen.flow_from_dataframe(
    dataframe=train_img_df,
    x_col='image_path',
    y_col='Pool_str',
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(64, activation='relu')(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=3
)

Found 12673 validated image filenames belonging to 2 classes.
Found 3168 validated image filenames belonging to 2 classes.


I0000 00:00:1789811745.312046 3045454 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4035 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/3


I0000 00:00:1789811750.835925 3045454 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1789811753.327665 3053007 service.cc:153] XLA service 0x77fa68032700 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1789811753.327683 3053007 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.22.0)
I0000 00:00:1789811753.406525 3053007 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1789811754.010065 3053007 cuda_dnn.cc:461] Loaded cuDNN version 92200
I0000 00:00:1789811754.038060 3053007 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8022__.114
E0000 00:00:1789811760.223597 3053007 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup e

  1/397 ━━━━━━━━━━━━━━━━━━━━ 1:21:50 12s/step - accuracy: 0.6250 - loss: 0.7643

I0000 00:00:1789811763.174179 3053007 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


115/397 ━━━━━━━━━━━━━━━━━━━━ 53s 189ms/step - accuracy: 0.6364 - loss: 0.6590

E0000 00:00:1789811786.454142 3053009 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


397/397 ━━━━━━━━━━━━━━━━━━━━ 110s 246ms/step - accuracy: 0.6809 - loss: 0.5865 - val_accuracy: 0.7004 - val_loss: 0.5644
Epoch 2/3
397/397 ━━━━━━━━━━━━━━━━━━━━ 84s 213ms/step - accuracy: 0.7183 - loss: 0.5366 - val_accuracy: 0.7247 - val_loss: 0.5432
Epoch 3/3
397/397 ━━━━━━━━━━━━━━━━━━━━ 87s 219ms/step - accuracy: 0.7362 - loss: 0.5081 - val_accuracy: 0.7175 - val_loss: 0.5534


In [11]:
from sklearn.metrics import  f1_score, classification_report, confusion_matrix
val_generator.reset()
preds = model.predict(val_generator)
y_pred = (preds > 0.5).astype(int).flatten()

y_true = val_generator.classes

print("F1 Score:", f1_score(y_true, y_pred))
print(classification_report(y_true, y_pred))


99/99 ━━━━━━━━━━━━━━━━━━━━ 19s 173ms/step
F1 Score: 0.7257125344774747
              precision    recall  f1-score   support

           0       0.63      0.80      0.71      1354
           1       0.82      0.65      0.73      1814

    accuracy                           0.72      3168
   macro avg       0.73      0.73      0.72      3168
weighted avg       0.74      0.72      0.72      3168



<h2 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
 Prediction for Test Data and Output
</font>
</h2>

<p dir=ltr style="direction: ltr;text-align: left;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    After feature engineering and modeling, you have an algorithm that can lead you from independent variables to the target variable.
    <br>
    Use this model to predict the samples present in the test data and prepare the results in the format of the table (<code>dataframe</code>) below.
</font>
</p>

<div dir=ltr style="direction: ltr;text-align: center;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    
|Column|Description|
|------|---|
|Pool|Presence or absence of a pool|
|id|Unique identifier of the accommodation|
    
</font>
</div>

<p dir="ltr" style="direction: ltr; text-align: justify; line-height: 200%; font-family: Vazir; font-size: medium;">
    The dataframe name must be <i>submission</i>; otherwise, the judging system cannot evaluate your attempt.
    <br>
    This dataframe contains two columns named <i>id</i> and <i>Pool</i> and has 862 rows.
    <br>
    The values in the <i>Pool</i> column are <i>True</i> or <i>False</i>.
    <br>
    For each row present in the <i>test</i> dataframe, you must have a predicted value.
    <br>
    The table below shows the first 5 rows of the <code>submission</code> dataframe. Of course, in your answer, the values of the <i>Pool</i> column might be different.
</p>

<div style="text-align: center; line-height: 200%; font-family: Vazir; font-size: medium;">
    <font face="vazir" size="3">
        <table border="1" style="margin: 0 auto;">
            <tr>
                <th>id</th>
                <th>Pool</th>
            </tr>
            <tr>
                <td>3160664</td>
                <td>True</td>
            </tr>
            <tr>
                <td>3195184</td>
                <td>False</td>
            </tr>
            <tr>
                <td>3224078</td>
                <td>False</td>
            </tr>
            <tr>
                <td>3233712</td>
                <td>True</td>
            </tr>
            <tr>
                <td>3201449</td>
                <td>True</td>
            </tr>
        </table>
    </font>
</div>

In [14]:
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_img_df,
    x_col='image_path',
    y_col=None,
    target_size=(128, 128),
    batch_size=32,
    class_mode=None,
    shuffle=False
)

test_preds = model.predict(test_generator)
test_img_df['pred_prob'] = test_preds

agg_preds = test_img_df.groupby('id')['pred_prob'].mean().reset_index()

agg_preds['pool'] = agg_preds['pred_prob'] > 0.5

submission = pd.DataFrame({'id': df_test['id']})
submission = submission.merge(agg_preds[['id', 'pool']], on='id', how='left')

submission['pool'] = submission['pool'].fillna(False)

submission['pool'] = submission['pool'].astype(bool)

print("Columns in submission:", submission.columns.tolist())
print(submission.head())

Found 8505 validated image filenames.
266/266 ━━━━━━━━━━━━━━━━━━━━ 49s 182ms/step
Columns in submission: ['id', 'pool']
        id   pool
0  3160664   True
1  3195184  False
2  3224078   True
3  3233712   True
4  3201449   True


In [ ]:
if not os.path.exists(os.path.join(os.getcwd(), 'EstakhtemoonNasheh.ipynb')):
    %notebook -e EstakhtemoonNasheh.ipynb

def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)
            
submission.to_csv('submission.csv', index=False)
file_names = ['EstakhtemoonNasheh.ipynb', 'submission.csv']
compress(file_names)

File Paths:
['EstakhtemoonNasheh.ipynb', 'submission.csv']
